In [ ]:
import torch
from torch import nn
from torch.nn import functional as F
import torch.nn as nn 
net=nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,20))
x=torch.rand(2,20)
net(x)

tensor([[-0.0481,  0.0722,  0.0690,  0.1269, -0.1055, -0.0984,  0.2398, -0.2435,
         -0.0807,  0.0138, -0.2352,  0.1382, -0.1812,  0.0162,  0.0106, -0.1331,
          0.1141, -0.0915,  0.0737,  0.1458],
        [-0.0956, -0.0315,  0.2166, -0.0073, -0.0737, -0.0899,  0.3152, -0.3495,
         -0.0648, -0.1763, -0.2944,  0.0394, -0.0990,  0.0779,  0.0961, -0.2443,
          0.0813, -0.1791,  0.1440,  0.1986]], grad_fn=<AddmmBackward0>)

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)
    
    def forward(self, x):
        return self.out(F.relu(self.hidden(x)))

In [ ]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for block in args:
            self._modules[block]=block
    def forward(self,x):
        for block in self._modules.values():
            x=block(x)
        return x
net=MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(x)


tensor([[ 0.0192,  0.0016,  0.1080,  0.0942,  0.1931,  0.1024, -0.4421, -0.2244,
          0.0868, -0.0263],
        [-0.0380, -0.0635, -0.0502,  0.2288,  0.1554,  0.0770, -0.3645, -0.2342,
         -0.0428, -0.0879]], grad_fn=<AddmmBackward0>)

In [7]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)  # 固定权重，不参与训练
        self.linear = nn.Linear(20, 20)
    
    def forward(self, X):
        X = self.linear(X)  # 注意：这里是大写 X
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X = X / 2  # 注意：X/2 要赋值回去
        return X.sum()
net=FixedHiddenMLP()
net(x)

tensor(0.3664, grad_fn=<SumBackward0>)